<a href="https://colab.research.google.com/github/rcsb/rcsb-training-resources/blob/master/training-events/2026/python-rcsb-api/sequence_api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Using `rcsb-api` to access RCSB PDB's Sequence Coordinates API

In [ ]:
# Install `rcsb-api`
%pip install --upgrade rcsb-api

# Install `rich` for pretty printing
%pip install rich

## Sequence Coordinate API Query Basics
There are two basic types of queries you can perform with the sequence coordinates API—`Alingments` and `Annotations`. 

- `Alignments` allows you to get sequence alignments and coverage between a PDB entity or instance, UniProt sequence, and/or NCBI genome or protein sequence.
- `Annotations` allows you to get positional features/annotations for a given PDB entity or instance, UniProt sequence, or NCBI genome or protein sequence. (Annotations are sourced from PDB data, UniProt, CATH, ECOD, SCOPe, and other third-party resources integrated into RCSB PDB data.)

### Arguments
Each type of query has similar but slightly different argument names and options; however, both types make use of the same set of possible values to provide to certain arguments, specifically those defining the ID of the sequence to query (following the format of the Examples in the table below) and the corresponding sequence data source for that ID (`SequenceReference` in the table below).

The table below describes the type of database identifiers used for each `SequenceReference` value.

| `SequenceReference` | Database Identifier Description              | Example                        |
|---------------------|-----------------------------------------------|--------------------------------|
| `NCBI_GENOME`       | NCBI RefSeq Chromosome Accession              | `NC_000001`                    |
| `NCBI_PROTEIN`      | NCBI RefSeq Protein Accession                 | `NP_789765`                    |
| `UNIPROT`           | UniProt Accession                             | `P01112`                       |
| `PDB_ENTITY`        | RCSB PDB Entity Id / CSM Entity Id            | `2UZI_3` / `AF_AFP68871F1_1`   |
| `PDB_INSTANCE`      | RCSB PDB Instance Id / CSM Instance Id        | `2UZI.C` / `AF_AFP68871F1.A`   |


For the complete list of possible arguments for both types of Sequence API queries, see the [documentation](https://rcsbapi.readthedocs.io/en/latest/seq_api/quickstart.html#getting-started).


## Creating a Sequence Alignment API Query

We'll start by creating a Sequence Alignment API query to determine the residue positions and coverage of an alignment.

1. This first query will fetch the sequence alignments between PDB entity 3V85_1 and its corresponding UniProt sequence: 

In [ ]:
from rcsbapi.sequence import Alignments
from rich import print as rprint

query = Alignments(
    db_from="PDB_ENTITY",
    db_to="UNIPROT",
    query_id="3V85_1",
    return_data_list=["query_sequence", "target_alignments", "alignment_length"]  # This specifies what data items to return
)
result_dict = query.exec()
rprint(result_dict)

2. Next, let's get the alignment between the same entity and its corresponding NCBI Genome sequence

In [ ]:
from rcsbapi.sequence import Alignments
from rich import print as rprint

query = Alignments(
    db_from="PDB_ENTITY",
    db_to="NCBI_GENOME",
    query_id="3V85_1",
    return_data_list=["query_sequence", "target_alignments", "alignment_length"]
)
result_dict = query.exec()
rprint(result_dict)

3. Last, rather than getting the alignment one entity at-a-time, we can request the sequence alignment of *all* PDB entities of UniProt Q9SIY3

In [ ]:
from rcsbapi.sequence import Alignments
from rich import print as rprint

query = Alignments(
    db_from="UNIPROT",
    db_to="PDB_ENTITY",
    query_id="Q9SIY3",
    return_data_list=["query_sequence", "target_alignments", "alignment_length"]
)
result_dict = query.exec()
rprint(result_dict)

## Creating a Sequence Annotations API Query

Next, we'll look at creating a Sequence Annotations API query to fetch positional annotations associated to the queried sequence.

1. First, we will fetch the sequence annotations for PDB instance 3V85.A

In [ ]:
from rcsbapi.sequence import Annotations

# Fetch all positional features for a particular PDB Instance
query = Annotations(
    reference="PDB_INSTANCE",
    query_id="3V85.A",
    sources=["UNIPROT", "PDB_ENTITY", "PDB_INSTANCE"],
    return_data_list=["target_id", "source", "features"]
)
result_dict = query.exec()
print(len(result_dict["data"]["annotations"]))

2. Now, we'll do the same query, but apply a filter to only return the `BINDING_SITE` anntotations

In [ ]:
from rcsbapi.sequence import Annotations, AnnotationFilterInput
from rich import print as rprint

# Fetch only binding_site features for a particular PDB Instance
query = Annotations(
    reference="PDB_INSTANCE",
    query_id="3V85.A",
    sources=["UNIPROT", "PDB_ENTITY", "PDB_INSTANCE"],
    return_data_list=["target_id", "source", "features"],
    filters=[
        AnnotationFilterInput(
            field="TYPE",
            operation="EQUALS",
            values=["BINDING_SITE"],
        )
    ],
)
result_dict = query.exec()
rprint(result_dict)

3. Last, let's get the NCBI Genome positions corresponding to the protein binding sites, for the entire UniProt Q9SIY3

In [ ]:
from rcsbapi.sequence import Annotations, AnnotationFilterInput

# Fetch only binding_site features for a particular PDB Instance
query = Annotations(
    reference="UNIPROT",
    query_id="Q9SIY3",
    sources=["PDB_INSTANCE"],  # Fetch only binding site annotations from the PDB_INSTANCE data source
    return_data_list=["target_id", "source", "features"],
    filters=[
        AnnotationFilterInput(
            field="TYPE",
            operation="EQUALS",
            values=["BINDING_SITE"],
        )
    ],
)
result_dict = query.exec()
print(len(result_dict["data"]["annotations"]))

## Visualizing and Manipulating Queries
One helpful feature available is the ability to view your query in our Sequence Coordinates API GraphiQL query editor, by using the `.get_editor_link()` method, e.g.:

In [ ]:
from rcsbapi.sequence import Alignments

query = Alignments(
    db_from="UNIPROT",
    db_to="PDB_ENTITY",
    query_id="Q9SIY3",
    return_data_list=["query_sequence", "target_alignments", "alignment_length"]
)
print(query.get_editor_link())

## Further Documentation
For more extensive examples and implementation details visit our [readthedocs](https://rcsbapi.readthedocs.io/en/latest/seq_api/quickstart.html).